# What Is Diffusion MRI?

Before touching any tool, you need to understand what the scanner is actually measuring. This chapter covers the physical basis of diffusion MRI and how the signal relates to white matter microstructure.

---

## 1. Water molecules never stop moving

At body temperature, water molecules undergo constant **Brownian motion** — random thermal motion in all directions. In a glass of water this motion is *isotropic*: equal in every direction. In the brain it is not.

White matter axons are wrapped in myelin sheaths and packed into bundles. Water molecules **inside or alongside these axons are restricted**: they can move more freely *along* the axis of the fibre than *perpendicular* to it. This anisotropy is the signal dMRI exploits.

```
  Isotropic (grey matter / CSF)      Anisotropic (white matter)

         ↑ ← ● → ↓                       ↑↑↑↑↑↑↑↑↑
         ↗ ↙ ↗ ↙                        ● ● ● ● ●  ← restricted
                                          ↓↓↓↓↓↓↓↓↓
```

## 2. How the scanner encodes diffusion

The Stejskal–Tanner pulse sequence adds two magnetic gradient pulses around the 180° refocusing pulse:

1. **First gradient** dephases proton spins — each proton accumulates a phase proportional to its position.
2. **Second gradient** (equal, opposite) tries to rephase them.
3. If a proton **moved** between the two pulses, it is at a different position → its phase is not fully cancelled → **signal loss**.

**More diffusion in the gradient direction → more signal loss.**

The amount of signal loss is governed by the **b-value**:

$$S = S_0 \, e^{-b \, D}$$

where $S_0$ is the unweighted signal (b=0), $D$ is the apparent diffusion coefficient along the gradient direction, and $b = \gamma^2 G^2 \delta^2 (\Delta - \delta/3)$.

## 3. The b-table: directions and shells

A single gradient direction only tells you about diffusion *in that direction*. To characterise the full 3-D diffusion profile you need many directions. The set of all gradient directions and b-values for a scan is called the **b-table** (or gradient table), stored in two files:

- **bvals**: one b-value per volume (e.g. `0 1000 1000 1000 ...`)
- **bvecs**: unit vectors, one per volume, as a 3×N matrix

HCP data has **three shells**: b=0, b=1000, b=2000, b=3000 s/mm² with 90 directions each.

## 4. What can go wrong (and what preprocessing fixes)

| Artefact | Cause | Fix |
|---|---|---|
| Thermal noise | Electronics | MP-PCA denoising |
| Gibbs ringing | k-space truncation | Partial Fourier removal |
| Eddy currents | Rapidly switched gradients | FSL eddy / dwifslpreproc |
| Head motion | Subject movement | FSL eddy (outlier replacement) |
| EPI distortion | B0 field inhomogeneity | topup (needs reverse-PE image) |
| Bias field | Coil sensitivity variation | FSL FAST / dwibiascorrect |

---

## 5. Hands-on: inspect a real HCP gradient table

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Paths (edit to match your data location) ─────────────────────────────────
data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
bvals_file = data_dir / 'bvals'
bvecs_file = data_dir / 'bvecs'

bvals = np.loadtxt(bvals_file)
bvecs = np.loadtxt(bvecs_file)   # shape: (3, N_volumes)

print(f'Total volumes : {len(bvals)}')
print(f'Unique shells : {np.unique(np.round(bvals, -2))}')  # round to nearest 100
print(f'b=0 volumes   : {np.sum(bvals < 50)}')
print(f'bvecs shape   : {bvecs.shape}')

Total volumes : 70
Unique shells : [   0. 1000. 2000.]
b=0 volumes   : 10
bvecs shape   : (3, 70)


In [ ]:
# Visualise the gradient directions on a sphere (one shell at a time)
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401

# Detect shells automatically — works with synthetic (b=1000/2000)
# and real HCP data (b=1000/2000/3000)
unique_bvals = np.unique(np.round(bvals, -2)).astype(int)
dw_shells    = unique_bvals[unique_bvals > 50]
palette      = ['royalblue', 'tomato', 'seagreen', 'purple']
shells       = {int(b): palette[i] for i, b in enumerate(dw_shells)}

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

for bval_target, color in shells.items():
    sel  = (bvals > bval_target - 100) & (bvals < bval_target + 100)
    dirs = bvecs[:, sel]
    ax.scatter(*dirs, s=20, c=color, alpha=0.8, label=f'b={bval_target}')
    ax.scatter(*(-dirs), s=20, c=color, alpha=0.3)   # antipodal symmetry

# Draw sphere wireframe
u, v = np.mgrid[0:2*np.pi:40j, 0:np.pi:20j]
ax.plot_wireframe(np.cos(u)*np.sin(v), np.sin(u)*np.sin(v), np.cos(v),
                  alpha=0.08, color='grey')

ax.set_title('Gradient directions per shell', fontsize=13)
ax.legend()
ax.set_box_aspect([1, 1, 1])
plt.tight_layout()
plt.show()
print('Well-distributed directions → more accurate diffusion model fitting')

In [ ]:
# Inspect the raw 4-D volume
import nibabel as nib

dwi_img  = nib.load(data_dir / 'data.nii.gz')
dwi_data = dwi_img.get_fdata()

print(f'4-D volume shape : {dwi_data.shape}')
print(f'Voxel size (mm)  : {dwi_img.header.get_zooms()[:3]}')
print(f'Data range       : {dwi_data.min():.0f} – {dwi_data.max():.0f}')
print(f'Available shells : {np.unique(np.round(bvals, -2)).astype(int)}')

# Pick b=0 and the highest available shell (works with any dataset)
b0_idx   = np.where(bvals < 50)[0][0]
high_idx = np.where(bvals == bvals.max())[0][0]
high_b   = int(round(bvals[high_idx], -2))

z = dwi_data.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, idx, label in zip(axes,
                           [b0_idx, high_idx],
                           [f'b=0  (vol {b0_idx})',
                            f'b≈{high_b}  (vol {high_idx})']):
    ax.imshow(dwi_data[:, :, z, idx].T, cmap='gray', origin='lower')
    ax.set_title(label)
    ax.axis('off')

fig.suptitle(f'Higher b-value → more diffusion weighting → darker image (more signal loss)',
             fontsize=11)
plt.tight_layout()
plt.show()

## Summary

- dMRI measures **restricted water diffusion** — the same water that moves more freely along fibres than across them.
- The scanner encodes diffusion via **gradient pulses** along many directions (the b-table).
- Higher b-value = stronger diffusion weighting = more signal loss, but also **more noise** and **more distortion**.
- HCP multi-shell data (b=1000/2000/3000) allows richer models than single-shell (e.g. CSD, NODDI) at the cost of longer scan time.

**Next**: [Tools overview →](../00_introduction/01_tools_overview.ipynb)